In [85]:
import pandas as pd
import numpy as np

In [86]:
df = pd.read_csv("IMDB Dataset.csv")

In [87]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [88]:
df.drop_duplicates(inplace=True)

### Pre processing

### Converting to lowercase

In [89]:
df["review"] = df["review"].str.lower()
df["sentiment"] = df["sentiment"].str.lower()

### Removing URLs

In [90]:
import re
def remove_urls(text):
    return re.sub(r"http\S+", "", text);

df["review"] = df["review"].apply(remove_urls)
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
...,...,...
49995,i thought this movie did a down right good job...,positive
49996,"bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,i am a catholic taught in parochial elementary...,negative
49998,i'm going to have to disagree with the previou...,negative


### Remove punctuations

In [91]:
def remove_punctuations(text):
    return re.sub(r"[^A-Za-z0-9\s]", "", text)

df["review"] = df["review"].apply(remove_punctuations)
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive
...,...,...
49995,i thought this movie did a down right good job...,positive
49996,bad plot bad dialogue bad acting idiotic direc...,negative
49997,i am a catholic taught in parochial elementary...,negative
49998,im going to have to disagree with the previous...,negative


### Removing HTML

In [92]:
def remove_html(text):
    return re.sub(r"<.*?>", "", text)

df["review"] = df["review"].apply(remove_html)

In [93]:
import nltk
# nltk.download("punkt")
# nltk.download("stopwords")
# nltk.download("punkt_tab")

### Stop words removal

In [94]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [95]:
def remove_stop_words(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")
    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")
    return text

df["review"] = df["review"].apply(remove_stop_words)

### Stemming

In [96]:
from nltk.stem import PorterStemmer

In [97]:
def stem(text):
    ps = PorterStemmer()
    stemmed_words = []
    tokens = word_tokenize(text=text)

    for token in tokens:
        stemmed_words.append(ps.stem(token))
    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stem)

### Encoding

In [98]:
from sklearn.preprocessing import LabelEncoder

In [99]:
le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

### Vectorization

In [100]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [101]:
tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])
Y = df["sentiment"]

X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057169 stored elements and shape (49582, 5000)>

# Datasets and Loaders

In [102]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42,
)

In [103]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader

In [104]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [105]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(Y_train.values).float()
)
test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(Y_test.values).float()
)

In [106]:
train_loader = DataLoader(
    train_set,
    shuffle=True,
    batch_size=64
)
test_loader = DataLoader(
    test_set,
    shuffle=True,
    batch_size=64
)

In [107]:
class RNN(nn.Module):

    def __init__(self, input_size,hidden_size = 128,num_layers = 1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            1
        )

    def forward(self,x):
        h0 = torch.zeros(
            self.num_layers,
            x.size(0),
            self.hidden_size
        )

        out, _ = self.rnn(x,h0)

        out = self.fc(out[:,-1,:])

        return out

In [108]:
import torch.optim as optim
input_size = X_train.shape[1]
model = RNN(input_size=input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(
    params=model.parameters()
)

In [ ]:
epochs = 10
best_val_loss = float('inf')

for epoch in range(epochs):
    model.train()
    epoch_train_loss = 0.0
    for xb,yb in train_loader:
        optimizer.zero_grad()

        xb = xb.unsqueeze(1)

        outputs = model(xb)
        outputs = torch.sigmoid(outputs.squeeze()) # 0 to 1

        loss =criterion(outputs,yb) # Compute Loss
        loss.backward() # Backward propogation
        optimizer.step() # Weights update
        
        epoch_train_loss +=loss.item()

    model.eval()

    running_val_loss = 0.0
    with torch.no_grad():
        for xb,yb in test_loader:
            xb = xb.unsqueeze(1)

            out = model(xb)
            out = torch.sigmoid(out.squeeze())
            loss = criterion(out,yb)
            running_val_loss+=loss.item()

    val_loss = running_val_loss/len(test_loader)

    if(val_loss<best_val_loss):
        best_val_loss= val_loss
        torch.save(model.state_dict(),"best_model.pt")
        print(f"Better Model Found at epoch #{epoch+1}")
    print(f"Epoch: {epoch+1}/{epochs} || Epoch Loss = {epoch_train_loss/len(train_loader)} || Val Loss = {val_loss}")


Better Model Found at epoch #1
Epoch: 1/10 || Epoch Loss = 0.37835413433851733 || Val Loss = 0.3002603535690615
Epoch: 2/10 || Epoch Loss = 0.26387242661608806 || Val Loss = 0.3058457979271489
Epoch: 3/10 || Epoch Loss = 0.24847765086639312 || Val Loss = 0.313337278942908
Epoch: 4/10 || Epoch Loss = 0.24098869922661012 || Val Loss = 0.3262876170296823
Epoch: 5/10 || Epoch Loss = 0.237004817998217 || Val Loss = 0.32502077923667044
Epoch: 6/10 || Epoch Loss = 0.23414788430015887 || Val Loss = 0.3288438316314451
Epoch: 7/10 || Epoch Loss = 0.23173969412042247 || Val Loss = 0.3298466070044425
Epoch: 8/10 || Epoch Loss = 0.23011085449928237 || Val Loss = 0.33445987965791457
Epoch: 9/10 || Epoch Loss = 0.2286944540998628 || Val Loss = 0.3387719006788346


In [ ]:
model.load_state_dict(torch.load("best_model.pt"))
model.eval()
with torch.no_grad():
    correct_vals = 0
    total = 0

    for xb,yb in test_loader:
        xb = xb.unsqueeze(1)
        outputs = model(xb)

        predicted = (torch.sigmoid((outputs.squeeze())) >0.5).float()

        total+=yb.size(0)

        correct_vals += (predicted == yb).sum().item()
    print(f"Accuracy: {correct_vals/total*100}")

In [ ]:


def clean(text):
    text= remove_html(text)
    text = remove_punctuations(text)
    text = remove_stop_words(text)
    text = remove_urls(text)
    text = stem(text)
    X = tf.transform([text])

    return X


In [2]:
def predict(text):
    
    text_vector = clean(text)
    tensor = torch.from_numpy(text_vector.toarray()).float().unsqueeze(1)
    with torch.no_grad():
        pred = model(tensor)
        prob = torch.sigmoid(pred).item()
        if(prob>0.5):
            sentiment = "Positive"
        elif(prob<0.4):
            sentiment = "Negative"
        else: 
            sentiment = "Neutral"

    print(f"Review: {text}")
    print(f"Probability: {prob}")
    print(f"Sentiment: {sentiment}\n")

In [3]:
samples = [
    "I don't like this product.",
    "I love this product!",
    "Idk"
]

for sample in samples:
    predict(sample)

NameError: name 'clean' is not defined